This notebook trains Support Vector Regression (SVR) models to predict hourly taxi demand based on temporal, holiday and POI features. Two spatial granularities are compared to identify which performs better.

Steps:
1. Load pre-split datasets (census tract & community area)
2. Explore data and target distributions
3. Prepare features (auto-detect, target encoding)
4. Train three SVR variants (Linear, RBF, Polynomial) with grid search
5. Evaluate and compare both datasets
6. Visualize predictions and metrics

> <b>[NOTE]</b> Weather data is not yet included and will be added in a later version.

## 0) Setup and configuration
All necessary packages are imported and the dataset paths are configured. The data is pre-split (train/test) and pre-scaled by the <i>03_data_preparation_for_nn_and_svm</i> notebook.

> <b>[NOTE]</b> H3 hexagonal cells were evaluated as a third spatial layer but are not used here: most hexagons contained zero or very few trips, i.e. extreme data sparsity, which would produce an unreliable model. The analysis that led to this decision is documented in the *Descriptive Analytics* notebook. Therefore only census tract and community area are compared below.

In [ ]:
# imports
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVR, SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    KFold, GridSearchCV,
)
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)

In [ ]:
# config: both datasets to compare
DATA_DIR = "../data/new_dataset_2026_06_16"
DATASETS = {
    "census_tract": {
        "train_path": f"{DATA_DIR}/03_census_tract_train.parquet",
        "test_path": f"{DATA_DIR}/03_census_tract_test.parquet",
        "spatial_col": "census_tract",
    },
    "community_area": {
        "train_path": f"{DATA_DIR}/03_community_area_train.parquet",
        "test_path": f"{DATA_DIR}/03_community_area_test.parquet",
        "spatial_col": "community_area",
    },
}

# 1) Data loading and exploration
Both datasets are loaded separately (train + test). The census tract dataset has significantly fewer records as approximately half of the census tracts were dropped during data cleaning due to missing values. We can also see that not all census tracts appear in the test set, while the community area dataset covers all 77 areas in both splits.

In [ ]:
# load both datasets (train + test separately) and show overview
raw_data = {}
for name, cfg in DATASETS.items():
    train_df = pl.read_parquet(cfg["train_path"])
    test_df = pl.read_parquet(cfg["test_path"])
    raw_data[name] = {"train": train_df, "test": test_df}
    spatial = cfg["spatial_col"]
    print(f"=== {name} ===")
    print(f"  Train:       {train_df.shape}")
    print(f"  Test:        {test_df.shape}")
    print(f"  Columns:     {train_df.columns}")
    print(f"  Unique {spatial} (train): {train_df[spatial].n_unique():,}")
    print(f"  Unique {spatial} (test):  {test_df[spatial].n_unique():,}")
    print(f"  trip_count (train): mean={train_df['trip_count'].mean():.2f}  "
          f"median={train_df['trip_count'].median():.1f}  max={train_df['trip_count'].max()}")
    print(f"  trip_count (test):  mean={test_df['trip_count'].mean():.2f}  "
          f"median={test_df['trip_count'].median():.1f}  max={test_df['trip_count'].max()}")
    nulls_train = sum(train_df.null_count().row(0))
    nulls_test = sum(test_df.null_count().row(0))
    print(f"  Nulls: train={nulls_train}  test={nulls_test}")
    print()

## 1.1) Target distribution
The `trip_count` distribution is right-skewed with a long tail, which is typical for demand data. Most hours have low trip counts while a few hours have very high demand.

In [ ]:
# compare trip_count distributions side by side (train data)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"census_tract": "#2980b9", "community_area": "#27ae60"}
for ax, (name, data) in zip(axes, raw_data.items()):
    ax.hist(data["train"]["trip_count"].to_numpy(), bins=80, color=colors[name], alpha=0.8)
    ax.set_title(f"trip_count Verteilung - {name} (train)")
    ax.set_xlabel("trip_count")
    ax.set_ylabel("Anzahl")
plt.tight_layout()
plt.show()

# 2) Feature preparation
Numeric features are auto-detected from both datasets. Boolean columns are cast to `Int64`. The spatial ID (`census_tract` or `community_area`) and the target `trip_count` are excluded from features.

A target encoding is applied: the mean `trip_count` per spatial unit is calculated on the training set only and joined back to both train and test. Unseen spatial units in the test set fall back to the global mean. This way the model receives information about the general demand level of each area without leaking test data.

> <b>[NOTE]</b> The pre-scaled data from notebook 03 contains <i>NaN</i> values in some POI columns (caused by <i>std=0</i> during scaling). These are filled with `0.0`, which is equivalent to the mean for standardized data.

In [ ]:
# helper: prepare pre-split data for SVR (target encoding + numpy extraction)
NUMERIC_DTYPES = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}

def prepare_dataset(train_df, test_df, spatial_col):
    """Target encoding (train only) + numpy arrays for pre-split, pre-scaled data."""
    # cast Boolean columns to Int64 so they work as numeric features
    bool_cols = [c for c in train_df.columns if train_df.schema[c] == pl.Boolean]
    if bool_cols:
        train_df = train_df.with_columns([pl.col(c).cast(pl.Int64) for c in bool_cols])
        test_df = test_df.with_columns([pl.col(c).cast(pl.Int64) for c in bool_cols])

    # auto-detect numeric feature columns (exclude spatial ID and target)
    exclude = {spatial_col, "trip_count"}
    feature_cols = [
        c for c in train_df.columns
        if c not in exclude and train_df.schema[c] in NUMERIC_DTYPES
    ]

    # fill NaN values (Polars NaN != null; caused by std=0 during scaling)
    for col in feature_cols:
        if train_df.schema[col] in (pl.Float32, pl.Float64):
            train_df = train_df.with_columns(pl.col(col).fill_nan(0.0))
            test_df = test_df.with_columns(pl.col(col).fill_nan(0.0))

    # target encoding: mean trip_count per spatial unit (train only)
    spatial_mean = (
        train_df.group_by(spatial_col)
        .agg(pl.col("trip_count").mean().alias("spatial_target_mean"))
    )
    global_mean = train_df["trip_count"].mean()
    train_df = train_df.join(spatial_mean, on=spatial_col, how="left").with_columns(
        pl.col("spatial_target_mean").fill_null(global_mean)
    )
    test_df = test_df.join(spatial_mean, on=spatial_col, how="left").with_columns(
        pl.col("spatial_target_mean").fill_null(global_mean)
    )

    feature_cols_final = feature_cols + ["spatial_target_mean"]

    X_train = train_df.select(feature_cols_final).to_numpy()
    y_train = train_df["trip_count"].to_numpy().astype(float)
    X_test = test_df.select(feature_cols_final).to_numpy()
    y_test = test_df["trip_count"].to_numpy().astype(float)

    return {
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "feature_cols": feature_cols_final,
        "n_features": len(feature_cols_final),
        "n_spatial": train_df[spatial_col].n_unique(),
    }

## Validation strategy
The train/test split was done **chronologically** in notebook 03: the records are ordered by timestamp and the most recent 20 % form the test set (~80 / 20 by time). This is essential for an honest demand-forecasting evaluation.

A random split would mix past and future observations and effectively turn the task into **temporal interpolation** rather than genuine *future* prediction - neighbouring rows would share the same weather/day-of-week constellation and the model could "cheat" instead of learning to extrapolate forward in time. The chronological split guarantees that the model is always evaluated on a later time period than it was trained on.

# 3) Model training
Three SVR variants are trained with grid search and 5-fold cross-validation:
- <b>LinearSVR</b>: fast linear baseline
- <b>RBF kernel</b>: captures non-linear relationships
- <b>Polynomial kernel</b>: flexible non-linear alternative

> <b>[NOTE]</b> Hyperparameter search uses `HalvingGridSearchCV` (Successive
> Halving) instead of an exhaustive `GridSearchCV`. The full parameter grid
> is still evaluated, but candidates compete on increasing subsets of the
> training sample: weak parameter combinations are eliminated early and only
> promising ones are refit on more data. This gives the same best-params
> result as a full grid search at a fraction of the runtime.

The best model is selected by the lowest cross-validated RMSE, then refit on the full training set and evaluated on the test set. A `StandardScaler` is kept inside the pipeline to scale the `spatial_target_mean` feature, which is the only unscaled input.

> <b>[NOTE]</b> Grid search uses a random subsample of 100,000 rows for efficiency. The final model is always refit on the entire training set.

In [ ]:
# helper: run 3 SVR models (Linear, RBF, Poly) with grid search,
# pick the best by CV RMSE, refit on full train, evaluate on test
GRID_SAMPLE_SIZE = 20000

def run_svr_pipeline(data, dataset_name):
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    # subsample for grid search
    np.random.seed(7)
    if len(X_train) > GRID_SAMPLE_SIZE:
        idx = np.random.choice(len(X_train), size=GRID_SAMPLE_SIZE, replace=False)
        X_sub, y_sub = X_train[idx], y_train[idx]
    else:
        X_sub, y_sub = X_train, y_train
    print(f"  Grid Search Sample: {X_sub.shape[0]:,} Zeilen ({data['n_features']} Features)")

    cv = KFold(n_splits=5, shuffle=True, random_state=7)

    # model configs: (name, pipeline, param_grid)
    configs = [
        ("LinearSVR",
         Pipeline([("scaler", StandardScaler()),
                   ("svr", LinearSVR(dual="auto", max_iter=10000, random_state=7))]),
         {"svr__C": [0.01, 0.1, 1, 10], "svr__epsilon": [0.1, 1, 10]}),
        ("RBF",
         Pipeline([("scaler", StandardScaler()),
                   # cache_size=2000 (MB) reduces redundant kernel evaluations during
                   # libsvm training. same result, faster fits
                   ("svr", SVR(kernel="rbf", max_iter=-1, cache_size=2000))]),
         {"svr__C": [0.1, 1, 10],
          "svr__gamma": ["scale", 0.01, 0.1, 1],
          "svr__epsilon": [0.1, 1, 10]}),
        ("Poly",
         Pipeline([("scaler", StandardScaler()),
                   ("svr", SVR(kernel="poly", max_iter=-1, cache_size=2000))]),
         {"svr__C": [0.1, 1, 10],
          "svr__degree": [2, 3],
          "svr__coef0": [0, 1],
          "svr__epsilon": [0.1, 1, 10]}),
    ]

    model_results = {}
    for model_name, pipe, grid in configs:
        print(f"  [{dataset_name}] {model_name} ...")
        gs = HalvingGridSearchCV(pipe, grid, cv=cv,
                                 scoring="neg_mean_squared_error",
                                 factor=3, resource="n_samples",
                                 min_resources="exhaust",
                                 n_jobs=-1, verbose=1, random_state=7)
        gs.fit(X_sub, y_sub)
        cv_rmse = np.sqrt(-gs.best_score_)
        model_results[model_name] = {
            "cv_rmse": cv_rmse,
            "params": gs.best_params_,
            "estimator": gs.best_estimator_,
        }
        print(f"    CV RMSE: {cv_rmse:.4f}  Params: {gs.best_params_}")

    # select best model by CV RMSE (lowest = best)
    best_name = min(model_results, key=lambda k: model_results[k]["cv_rmse"])
    best_model = model_results[best_name]["estimator"]
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # MAPE only on rows with actual > 0
    mask = y_test > 0
    mape = (np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
            if mask.sum() > 0 else float("nan"))

    print(f"\n  [{dataset_name}] Bestes Modell: {best_name}")
    print(f"    Test MAE:  {mae:.4f}")
    print(f"    Test RMSE: {rmse:.4f}")
    print(f"    Test R2:   {r2:.4f}")
    print(f"    Test MAPE: {mape:.2f}%")

    return {
        "best_model": best_name,
        "best_params": model_results[best_name]["params"],
        "cv_rmse": model_results[best_name]["cv_rmse"],
        "test_mae": mae,
        "test_rmse": rmse,
        "test_r2": r2,
        "test_mape": mape,
        "y_pred": y_pred,
        "y_test": y_test,
        "all_cv_rmse": {k: v["cv_rmse"] for k, v in model_results.items()},
    }

## 3.1) Run pipeline on both datasets
The full pipeline is executed for both census tract and community area datasets. Each dataset goes through feature preparation, grid search and evaluation independently.

In [ ]:
# quick speed test: 1 fit per model to estimate total runtime
import time

# prepare census_tract if not done yet
prepared = {}
if "census_tract" not in prepared:
    prepared["census_tract"] = prepare_dataset(
        raw_data["census_tract"]["train"], raw_data["census_tract"]["test"], "census_tract"
    )

# same subsample logic as grid search (one CV fold = 80% of subsample)
np.random.seed(7)
X_full = prepared["census_tract"]["X_train"]
y_full = prepared["census_tract"]["y_train"]
if len(X_full) > GRID_SAMPLE_SIZE:
    idx = np.random.choice(len(X_full), size=GRID_SAMPLE_SIZE, replace=False)
    X_sub, y_sub = X_full[idx], y_full[idx]
else:
    X_sub, y_sub = X_full, y_full
n_fold = int(len(X_sub) * 0.8)
X_fold, y_fold = X_sub[:n_fold], y_sub[:n_fold]

bench_models = [
    ("LinearSVR", Pipeline([("scaler", StandardScaler()),
                             ("svr", LinearSVR(C=1, dual="auto", max_iter=10000, random_state=7))])),
    ("RBF", Pipeline([("scaler", StandardScaler()),
                      ("svr", SVR(kernel="rbf", C=1, gamma="scale"))])),
    ("Poly", Pipeline([("scaler", StandardScaler()),
                       ("svr", SVR(kernel="poly", C=1, degree=2, coef0=0))])),
]
grid_fits = {"LinearSVR": 60, "RBF": 180, "Poly": 180}

print(f"Speed test: 1 fit per model on {n_fold:,} rows (one CV fold)")
print(f"GRID_SAMPLE_SIZE = {GRID_SAMPLE_SIZE:,}\n")
total_est = 0
for name, pipe in bench_models:
    t0 = time.time()
    pipe.fit(X_fold, y_fold)
    elapsed = time.time() - t0
    model_total = elapsed * grid_fits[name]
    total_est += model_total
    print(f"  {name:12s}  {elapsed:6.1f}s/Fit  ->  {grid_fits[name]} Fits = ~{model_total/60:.0f} Min/Dataset")
print(f"\n  Geschaetzte Gesamtzeit (2 Datasets): ~{total_est*2/60:.0f} Min")


In [ ]:
# run full pipeline on both datasets
all_results = {}
prepared = {}
for name, cfg in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {name}")
    print(f"{'='*60}")

    prepared[name] = prepare_dataset(
        raw_data[name]["train"], raw_data[name]["test"], cfg["spatial_col"]
    )
    p = prepared[name]
    print(f"  Features: {p['n_features']}  |  Unique spatial: {p['n_spatial']:,}")
    print(f"  Train: {p['X_train'].shape[0]:,}  |  Test: {p['X_test'].shape[0]:,}")
    print(f"  Feature columns: {p['feature_cols']}")

    all_results[name] = run_svr_pipeline(prepared[name], name)

# 4) Results
Below table compares the best model for each dataset across multiple metrics. The per-model breakdown shows the cross-validated RMSE for all three SVR variants, allowing us to assess which kernel and which spatial granularity performs best.

In [ ]:
# comparison table
rows = []
for name, res in all_results.items():
    rows.append({
        "Dataset": name,
        "Best Model": res["best_model"],
        "Features": prepared[name]["n_features"],
        "CV RMSE": round(res["cv_rmse"], 4),
        "Test MAE": round(res["test_mae"], 4),
        "Test RMSE": round(res["test_rmse"], 4),
        "Test R2": round(res["test_r2"], 4),
        "Test MAPE": round(res["test_mape"], 2),
    })
comparison = pl.DataFrame(rows)
display(comparison)

# per-dataset breakdown of all 3 models
print("\nCV RMSE pro Modell:")
for name, res in all_results.items():
    for model, rmse in res["all_cv_rmse"].items():
        marker = " <-- BEST" if model == res["best_model"] else ""
        print(f"  {name:16s} {model:12s} {rmse:.4f}{marker}")

# 5) Visualization
## 5.1) Predicted vs Actual
Scatter plots compare predicted against actual `trip_count` values for each dataset. The red dashed line represents a perfect prediction. Points below the line indicate underestimation, points above indicate overestimation.

In [ ]:
# comparison plots: predicted vs actual for each dataset
n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 6), squeeze=False)
colors = {"census_tract": "#2980b9", "community_area": "#27ae60"}

for ax, (name, res) in zip(axes[0], all_results.items()):
    y_t, y_p = res["y_test"], res["y_pred"]
    ax.scatter(y_t, y_p, s=8, alpha=0.3, color=colors.get(name, "#2980b9"))
    lims = [min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())]
    ax.plot(lims, lims, "r--", lw=1.5, label="perfekte Vorhersage")
    ax.set_xlabel("Tatsaechlicher trip_count")
    ax.set_ylabel("Vorhergesagter trip_count")
    ax.set_title(f"{name} - {res['best_model']}\n(R2={res['test_r2']:.3f}, RMSE={res['test_rmse']:.2f})")
    ax.legend()

plt.tight_layout()
plt.show()

## 5.2) Metric comparison
The bar charts below compare the test metrics side by side. Lower MAE and RMSE indicate better predictions, while higher R2 indicates a better fit.

In [ ]:
# comparison bar chart: test metrics side by side
metrics = ["test_mae", "test_rmse", "test_r2"]
metric_labels = ["MAE", "RMSE", "R2"]
dataset_names = list(all_results.keys())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, label in zip(axes, metrics, metric_labels):
    vals = [all_results[n][metric] for n in dataset_names]
    bars = ax.bar(dataset_names, vals, color=[colors.get(n, "#2980b9") for n in dataset_names])
    ax.set_title(f"Test {label}")
    ax.set_ylabel(label)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{v:.3f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

# Shortfalls & Improvement Levers

**Shortfalls of the current SVR setup**
- *Right-skewed target*: `trip_count` has a long right tail, so the model is dominated by a few high-demand rows and the squared-error objective over-weights them.
- *Sparse spatial units*: many census tracts contain only a handful of trips, which makes per-unit predictions noisy and unstable.
- *Missing weather features*: temperature, precipitation, etc. were collected earlier but are **not** present in the final modelling dataset used here, although weather strongly drives taxi demand.
- *SVR scaling limitation*: kernel SVR (RBF/Poly) scales roughly quadratically with the number of samples, so grid search had to rely on a small subsample (`GRID_SAMPLE_SIZE = 20 000`) - the full training set (>300 k rows) is intractable for non-linear SVR.
- *Only two spatial layers*: H3 hexagons were dropped because of extreme sparsity (see *Descriptive Analytics*), so the spatial comparison is limited to census tract vs. community area.

**Improvement levers**
- *Weather integration*: add the already-collected weather features as model inputs.
- *Target transform*: apply a `log1p` transform to `trip_count` (and invert for evaluation) to reduce skew and stabilise the variance.
- *Spatial features*: add POI density, distance to the CBD / nearest transit hub, and neighbourhood embeddings so the model gets richer location context.
- *Stronger learners*: replace/augment SVR with gradient-boosted trees (e.g. LightGBM / XGBoost), which handle non-linearities, large n and feature interactions far more efficiently.
- *Spatial cross-validation*: use grouped / spatial CV (group by spatial unit and time block) so that nearby / overlapping observations do not leak between folds, giving a more honest error estimate.